# 🧢 Detector de Acessórios de Cabeça em Fotos de Candidatos (Google Colab)

Este notebook executa a detecção de **acessórios de cabeça** (chapéus, bonés, toucas, tiaras, capacetes, turbantes, etc.) em fotos de candidatos com limiar de **confiança estrito superior a 90% (>0.90)**.

---

## 🛠️ Passo 1: Instalação das Bibliotecas Necessárias no Google Colab

In [ ]:
# Instalar as dependências do YOLO-World, CLIP da Ultralytics e dependências auxiliares
!pip install -q ultralytics transformers pillow pandas ftfy git+https://github.com/ultralytics/CLIP.git

## 📁 Passo 2: Extrair Fotos (Amostra do GitHub ou foto_cand2024_SP_div.zip do Drive)

Escolha uma das opções abaixo no Colab. A **Opção B** detecta automaticamente o arquivo `foto_cand2024_SP_div.zip` na pasta do seu Google Drive e descompacta no SSD local.

In [ ]:
# OPÇÃO A: Extrair fotos de amostra diretamente do GitHub (Recomendado para testes rápidos)
import os
import shutil

!git clone https://github.com/koiti/fotos_candidatos.git

if os.path.exists('fotos_candidatos/amostras'):
    if os.path.exists('amostras'):
        shutil.rmtree('amostras')
    shutil.copytree('fotos_candidatos/amostras', 'amostras')
    print("-> Fotos de amostra extraídas com sucesso do GitHub para a pasta 'amostras'!")

In [ ]:
# OPÇÃO B: Processar Dataset Grande (foto_cand2024_SP_div.zip com 78k fotos e 2GB) via Google Drive
# DICA DE DESEMPENHO: O código busca o arquivo .zip no seu Drive (na mesma pasta do notebook ou subpastas),
# copia para o SSD local da máquina do Colab em segundos e descompacta lá. Isso deixa o I/O 50x mais rápido!
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')

zip_name = 'foto_cand2024_SP_div.zip'
zip_found = None

# Verificar locais comuns no Google Drive
search_paths = [
    Path('/content/drive/MyDrive') / zip_name,
    Path('/content/drive/MyDrive/UFABC/2026-TOPICOS_DE_IA') / zip_name,
    Path('/content/drive/MyDrive/2026-TOPICOS_DE_IA') / zip_name
]

for p in search_paths:
    if p.exists():
        zip_found = p
        break

if not zip_found:
    print(f"🔍 Procurando '{zip_name}' no seu Google Drive...")
    matches = list(Path('/content/drive/MyDrive').rglob(zip_name))
    if matches:
        zip_found = matches[0]

if zip_found:
    print(f"-> Arquivo localizado com sucesso: {zip_found}")
    print("-> Copiando .zip do Drive para o SSD ultrarrápido do Colab...")
    !cp "{zip_found}" /content/
    print("-> Descompactando 78.000 fotos no SSD local...")
    !unzip -q /content/foto_cand2024_SP_div.zip -d /content/foto_cand2024_SP
    print("-> SUCESSO! 78.000 fotos prontas para análise ultrarrápida na pasta '/content/foto_cand2024_SP'!")
else:
    print(f"⚠️ Arquivo '{zip_name}' não encontrado no seu Google Drive. Verifique se o upload foi concluído.")

In [ ]:
# OPÇÃO C: Fazer upload direto de um arquivo ZIP contendo as fotos (ex: amostras.zip)
from google.colab import files
import zipfile

print('Faça o upload do seu arquivo .zip com as fotos:')
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('amostras')
        print("-> Arquivos extraídos com sucesso na pasta 'amostras'!")

## 🧠 Passo 3: Código do Detector de Acessórios de Cabeça (YOLO-World Open-Vocabulary)

In [ ]:
import os
os.environ['YOLO_AUTOINSTALL'] = 'False'

import json
import time
import torch
from pathlib import Path
from datetime import datetime
from PIL import Image, ImageDraw, ImageOps
from ultralytics import YOLO
from IPython.display import display, Image as IPImage
import pandas as pd

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo de inferência no Colab: {device.upper()}')

CLASSES_CABETA_MAP = {
    'hat': 'chapéu',
    'cap': 'boné',
    'baseball cap': 'boné',
    'headband': 'tiara/faixa',
    'head covering': 'cobertura de cabeça',
    'hood': 'capuz',
    'bonnet': 'touca',
    'turban': 'turbante',
    'helmet': 'capacete',
    'beret': 'boina',
    'beanie': 'gorro'
}
PROMPTS = list(CLASSES_CABETA_MAP.keys())

def executar_deteccao_acessorios_cabeca_colab(
    input_dir='amostras',
    output_dir='irregular_acessorios_cabeca',
    conf_thresh=0.90,
    batch_size=64
):
    target_input = Path(input_dir)
    if not target_input.exists() and Path('/content/foto_cand2024_SP').exists():
        target_input = Path('/content/foto_cand2024_SP')

    target_output = Path(output_dir)
    target_output.mkdir(exist_ok=True, parents=True)

    fotos = sorted([
        f for f in target_input.iterdir()
        if f.is_file() and f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}
    ])

    if not fotos:
        print(f"Nenhuma foto encontrada na pasta '{target_input}'!")
        return None

    print(f"Analizando {len(fotos)} fotos em '{target_input}' com limiar de {conf_thresh:.0%}...")
    print('Carregando modelo YOLO-World...')
    model = YOLO('yolov8s-worldv2.pt')
    model.set_classes(PROMPTS)

    total_irregulares = 0
    resultados_detalhados = []
    t_start = time.time()

    for b_idx in range(0, len(fotos), batch_size):
        bfiles = fotos[b_idx:b_idx + batch_size]
        bimgs = []
        vfiles = []
        for f in bfiles:
            try:
                img = Image.open(f)
                img = ImageOps.exif_transpose(img).convert('RGB')
                bimgs.append(img)
                vfiles.append(f)
            except Exception:
                pass

        if not bimgs:
            continue

        results = model.predict(source=bimgs, conf=conf_thresh, batch=batch_size, device=device, verbose=False)

        for foto_path, img_rgb, res in zip(vfiles, bimgs, results):
            irregularidades = []
            if len(res.boxes) > 0:
                boxes = res.boxes.xyxy.cpu().numpy()
                confs = res.boxes.conf.cpu().numpy()
                clss = res.boxes.cls.cpu().numpy().astype(int)

                for box, conf, cls_idx in zip(boxes, confs, clss):
                    c_val = float(conf)
                    if c_val >= conf_thresh:
                        lbl_en = PROMPTS[cls_idx] if cls_idx < len(PROMPTS) else 'head accessory'
                        lbl_pt = CLASSES_CABETA_MAP.get(lbl_en, lbl_en)
                        irregularidades.append({
                            'classe_en': lbl_en,
                            'classe_pt': lbl_pt,
                            'confianca': round(c_val, 4),
                            'bbox': [round(float(c), 2) for c in box]
                        })

            if irregularidades:
                total_irregulares += 1
                draw = ImageDraw.Draw(img_rgb)
                for det in irregularidades:
                    box = det['bbox']
                    label_pt = det['classe_pt']
                    conf = det['confianca']
                    draw.rectangle(box, outline='red', width=4)
                    draw.text((box[0] + 5, max(0, box[1] - 15)), f'{label_pt} ({conf:.1%})', fill='red')
                img_rgb.save(target_output / foto_path.name)
                print(f" -> DETECTADO: {foto_path.name} | {irregularidades[0]['classe_pt']} ({irregularidades[0]['confianca']:.1%})")

            resultados_detalhados.append({
                'arquivo': foto_path.name,
                'status': 'irregular' if irregularidades else 'regular',
                'irregularidades': irregularidades
            })

    t_total = time.time() - t_start
    relatorio_data = {
        'data_analise': datetime.now().isoformat(),
        'limiar_confianca': conf_thresh,
        'total_fotos': len(fotos),
        'total_irregulares': total_irregulares,
        'tempo_execucao_segundos': round(t_total, 2),
        'resultados': resultados_detalhados
    }

    relatorio_path = target_output / 'relatorio.json'
    with open(relatorio_path, 'w', encoding='utf-8') as f:
        json.dump(relatorio_data, f, ensure_ascii=False, indent=2)

    print('\n' + '=' * 70)
    print('VARREDURA CONCLUÍDA NO COLAB!')
    print(f'Total de fotos analisadas: {len(fotos)}')
    print(f'Irregulares salvas em "{target_output}": {total_irregulares}')
    print(f'Relatório salvo em: {relatorio_path}')
    return relatorio_data

## 🚀 Passo 4: Executar a Detecção

In [ ]:
# Executar a detecção com limiar de 90% de confiança
# Por padrão usa 'amostras' se existir, ou '/content/foto_cand2024_SP' se o dataset do Drive foi extraído
pasta_fotos = '/content/foto_cand2024_SP' if os.path.exists('/content/foto_cand2024_SP') else 'amostras'

resultado = executar_deteccao_acessorios_cabeca_colab(
    input_dir=pasta_fotos,
    output_dir='irregular_acessorios_cabeca',
    conf_thresh=0.90
)

# Exibir dataframe com resultados irregulares
if resultado:
    df = pd.DataFrame(resultado['resultados'])
    display(df[df['status'] == 'irregular'])

## 🖼️ Passo 5: Visualizar Imagens Irregulares no Colab

In [ ]:
# Exibir fotos anotadas com bounding box vermelha
target = Path('irregular_acessorios_cabeca')
fotos_irregulares = sorted([f for f in target.glob('*.[jJ][pP]*[gG]')])

if fotos_irregulares:
    print(f'Exibindo {len(fotos_irregulares)} fotos irregulares:')
    for f in fotos_irregulares[:20]:  # Exibe até as primeiras 20 para visualização
        print(f'📷 {f.name}')
        display(IPImage(filename=str(f), width=350))
else:
    print('Nenhuma imagem irregular encontrada com mais de 90% de certeza!')

## 💾 Passo 6: Baixar Resultados (.zip)

In [ ]:
# Compactar a pasta de resultados e fazer download para seu computador
!zip -r irregular_acessorios_cabeca.zip irregular_acessorios_cabeca
from google.colab import files
files.download('irregular_acessorios_cabeca.zip')